# VLA Usage Instructions

This documentation explains how to setup SmolVLA/LiteVLA. It uses the [Omega1029/offline-robotcs](https://github.com/Omega1029/offline-robotcs) repository,**HuggingFaceTB/SmolVLM-Base**, and the `smolvlm_turtlebot_action_balanced` adapter to predict text actions given by the user.

**Use this notebook on the Jetson device.** Copy the Bash commands into a terminal. Run the Python cells using the Jetson's Python (SmolVLA Jetson) kernel. Camera, inference, and controller processes each need their own terminal to run on.

The commands below assume the user has Ubuntu 24.04 with ROS 2 Jazzy. Change the workspace path if yours differs.

| File in `src/turtlebot3_smolvla/turtlebot3_smolvla/` | Purpose |
|---|---|
| `vla_model.py` | Loads the model and adapter; predicts one action from an image. |
| `inference_node.py` | Reads camera frames, publishes predictions, and detects STOP signs using OpenCV and Tesseract. |
| `action_controller.py` | Handles enable/disable, robot velocity, and the repeating square demo. |

Keep the existing package files. There is no need to recreate these scripts.







# 1. Prepare the Jetson Device

Confirm the system has the necessary installations before choosing a PyTorch build:

```bash
cat /etc/os-release
cat /etc/nv_tegra_release
/usr/bin/python3 --version
ls /opt/ros
```

If ROS 2 Jazzy is missing, follow the [ROS installation instructions](https://docs.ros.org/en/jazzy/Installation/Ubuntu-Install-Debs.html) first. Then install the dependencies:

```bash
sudo apt update
sudo apt install -y \
  git git-lfs python3-venv python3-pip python3-colcon-common-extensions \
  python3-numpy python3-opencv python3-pil \
  tesseract-ocr tesseract-ocr-eng v4l-utils \
  ros-jazzy-rclpy ros-jazzy-cv-bridge \
  ros-jazzy-sensor-msgs ros-jazzy-std-msgs \
  ros-jazzy-geometry-msgs ros-jazzy-nav-msgs ros-jazzy-v4l2-camera
```

These explicit installs include runtime dependencies missing from the package metadata, including `nav_msgs` and the Tesseract executable.

## 2. Create the Python environment

Use the system Python so ROS packages remain accessible:

```bash
source /opt/ros/jazzy/setup.bash
/usr/bin/python3 -m venv --system-site-packages ~/venvs/smolvla
source ~/venvs/smolvla/bin/activate
python -m pip install --upgrade pip
python -m pip install 'setuptools==79.0.1' wheel
```

Check PyTorch in this environment:

```bash
python -c 'import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())'
```

The final value must be `True`. If PyTorch is missing or CUDA is unavailable, install a CUDA-enabled build matching the Jetson's JetPack/L4T and Python versions using [NVIDIA's Jetson PyTorch guide](https://docs.nvidia.com/deeplearning/frameworks/install-pytorch-jetson-platform/index.html). Install it into this environment unless a compatible system installation is already visible. The folder does not identify a verified wheel URL.

Once CUDA works, preserve that PyTorch version while installing the remaining packages:

```bash
python -c 'import torch; from pathlib import Path; Path.home().joinpath("venvs/smolvla/constraints.txt").write_text("torch==" + torch.__version__ + "\nsetuptools==79.0.1\nnumpy<2\n")'
python -m pip install -c ~/venvs/smolvla/constraints.txt \
  'transformers>=4.56,<5' 'peft>=0.18,<0.19' \
  accelerate pillow safetensors pytesseract ipykernel jupyterlab
python -m pip check
```

The constraints preserve the Jetson PyTorch build and the earlier setuptools/NumPy compatibility settings.


## 3. Get the model adapter

The loader expects this exact directory:
`~/offline-robotcs/smolvlm_turtlebot_action_balanced`.

If you do not already have the resources:

```bash
cd ~
git lfs install
git clone https://github.com/Omega1029/offline-robotcs.git
cd ~/offline-robotcs
git lfs pull
ls smolvlm_turtlebot_action_balanced
```

If the clone already exists, skip `git clone` and run `git lfs pull` inside it. The repository address comes from the previous notebook; its availability could not be verified during this revision. If unavailable, use the team's saved copy.

The adapter directory needs its configuration, actual adapter weights, and processor/tokenizer files. The base model downloads separately on first use.

## 4. Build the Supplied ROS Package

Transfer the supplied workspace's **src** directory to `~/ros2_ws/src" on the Jetson. The package root must be: 

```text
~/ros2_ws/src/turtlebot3_smolvla/package.xml
~/ros2_ws/src/turtlebot3_smolvla/setup.py
~/ros2_ws/src/turtlebot3_smolvla/setup.cfg
~/ros2_ws/src/turtlebot3_smolvla/resource/turtlebot3_smolvla
~/ros2_ws/src/turtlebot3_smolvla/turtlebot3_smolvla/
```

Build on the Jetson using its own environment. The supplied `build`, `install`, and `log` directories are generated files; do not reuse another machine's copies.

```bash
source /opt/ros/jazzy/setup.bash
source ~/venvs/smolvla/bin/activate
cd ~/ros2_ws
python "$(command -v colcon)" build --symlink-install --packages-select turtlebot3_smolvla
source ~/ros2_ws/install/setup.bash
ros2 pkg executables turtlebot3_smolvla
```

The executable list should include **smolvla_inference** and **action_controller**. Only this package is needed for the guide; the other packages in `src` are separate demos.

## 5. Open Jupyter and Check the Environment

Run from the directory containing this notebook: 

```bash
source /opt/ros/jazzy/setup.bash
source ~/venvs/smolvla/bin/activate
source ~/ros2_ws/install/setup.bash
python -m ipykernel install --user --name smolvla --display-name 'Python (SmolVLA Jetson)'
python -m jupyterlab
```

Open the notebook and select **Python (SmolVLA Jetson)**. Run the next cell; resolve failures before continuing.

In [ ]:
import importlib
import shutil
import sys
import torch

print("Python:", sys.executable)
for module in ["transformers", "peft", "PIL", "cv2", "numpy", "pytesseract",
               "rclpy", "cv_bridge", "sensor_msgs.msg", "std_msgs.msg",
               "geometry_msgs.msg", "nav_msgs.msg"]:
    importlib.import_module(module)

assert shutil.which("tesseract"), "Install tesseract-ocr."
assert torch.cuda.is_available(), "A CUDA-enabled Jetson PyTorch build is required."
probe = torch.ones((32, 32), device="cuda", dtype=torch.float16)
assert (probe @ probe)[0, 0].item() == 32.0
del probe
print("Imports and CUDA operation passed:", torch.cuda.get_device_name(0))